# Complete Metrics Analysis

Calculate all clinical metrics for the Multi-Agent Sepsis Predictor:
- AUROC & AUPRC (already have)
- Sensitivity, Specificity
- PPV (Precision), NPV
- F1 Score
- Confusion Matrix at various thresholds

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Libraries loaded!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration
MODEL_VERSION = "v3_large_lr1e4"  # Best model
DATA_PATH = "/content/drive/MyDrive/Sepsis/data/processed/mimic_harmonized"
MODEL_PATH = f"/content/drive/MyDrive/Sepsis/models/{MODEL_VERSION}"
OUTPUT_PATH = f"/content/drive/MyDrive/Sepsis/models/{MODEL_VERSION}"

# Features
VITALS_FEATURES = ['hr', 'resp', 'temp', 'sbp', 'dbp', 'map_value', 'o2sat']
LABS_FEATURES = ['bun', 'chloride', 'creatinine', 'wbc', 'bicarbonate', 'platelets',
                 'magnesium', 'calcium', 'potassium', 'sodium', 'glucose',
                 'fio2', 'ph', 'paco2', 'pao2', 'lactate', 'bilirubin']
ALL_FEATURES = VITALS_FEATURES + LABS_FEATURES

SEQUENCE_LENGTH = 24
RANDOM_SEED = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Model: {MODEL_VERSION}")
print(f"Device: {device}")

In [ ]:
# Add src to path and import model
import sys
sys.path.append("/content/drive/MyDrive/Sepsis/src")

from models.multi_agent import MultiAgentSepsisPredictor
print("Model class imported!")

## 1. Load Data and Model

In [ ]:
# Load data
print("Loading data...")
df = pd.read_hdf(f"{DATA_PATH}/mimic_processed_large.h5", key='data')
print(f"Loaded {len(df):,} observations")

In [ ]:
# Normalize features (same as training notebook cell 11)
print("Normalizing features...")
feature_stats = {}
for feature in ALL_FEATURES:
    mean = df[feature].mean()
    std = df[feature].std()
    if std == 0 or pd.isna(std):
        std = 1.0
    feature_stats[feature] = {'mean': float(mean), 'std': float(std)}
    df[feature] = (df[feature] - mean) / std

# Sanity check
print("Normalization check (should be near 0 mean, 1 std):")
for feat in ALL_FEATURES[:3]:
    print(f"  {feat}: mean={df[feat].mean():.3f}, std={df[feat].std():.3f}")

# Same train/test split as training (stratified)
from sklearn.model_selection import train_test_split

unique_patients = df['subject_id'].unique()
np.random.seed(RANDOM_SEED)
np.random.shuffle(unique_patients)

patient_labels = df.groupby('subject_id')['sepsis_label'].max().loc[unique_patients]

train_val_ids, test_ids = train_test_split(
    unique_patients,
    test_size=0.2,
    stratify=patient_labels,
    random_state=RANDOM_SEED
)

test_df = df[df['subject_id'].isin(test_ids)].copy()
print(f"\nTest set: {len(test_df):,} observations, {len(test_ids)} patients")

In [ ]:
# Dataset class (matching training notebook exactly)
from tqdm.notebook import tqdm

class SepsisSequenceDataset(Dataset):
    def __init__(self, df, vitals_features, labs_features, sequence_length=24):
        self.sequence_length = sequence_length
        self.vitals_features = vitals_features
        self.labs_features = labs_features
        self.all_features = vitals_features + labs_features
        
        self.sequences = []
        self.labels = []
        
        print("Creating sequences...")
        for patient_id, group in tqdm(df.groupby('subject_id')):
            group = group.sort_values('charttime').reset_index(drop=True)
            
            if len(group) < sequence_length:
                continue
            
            for i in range(sequence_length, len(group) + 1):
                seq = group.iloc[i-sequence_length:i]
                label = seq['sepsis_label'].iloc[-1]
                features = seq[self.all_features].values
                self.sequences.append(features)
                self.labels.append(label)
        
        self.sequences = np.array(self.sequences, dtype=np.float32)
        self.labels = np.array(self.labels, dtype=np.float32)
        
        # Create missing mask BEFORE filling NaN
        self.missing_mask = np.isnan(self.sequences)
        self.sequences = np.nan_to_num(self.sequences, nan=0.0)
        
        print(f"Created {len(self.sequences):,} sequences")
        print(f"Positive rate: {self.labels.mean()*100:.1f}%")
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        n_vitals = len(self.vitals_features)
        
        vitals = self.sequences[idx, :, :n_vitals]
        labs = self.sequences[idx, :, n_vitals:]
        labs_mask = self.missing_mask[idx, :, n_vitals:]
        all_features = self.sequences[idx]
        label = self.labels[idx]
        
        return {
            'vitals': torch.tensor(vitals, dtype=torch.float32),
            'labs': torch.tensor(labs, dtype=torch.float32),
            'labs_mask': torch.tensor(labs_mask, dtype=torch.float32),
            'all_features': torch.tensor(all_features, dtype=torch.float32),
            'label': torch.tensor(label, dtype=torch.float32)
        }

# Create test dataset
print("Creating test dataset...")
test_dataset = SepsisSequenceDataset(test_df, VITALS_FEATURES, LABS_FEATURES, SEQUENCE_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
print(f"Test sequences: {len(test_dataset):,}")

In [ ]:
# Load model
print("Loading model...")
model = MultiAgentSepsisPredictor(
    vitals_dim=len(VITALS_FEATURES),
    labs_dim=len(LABS_FEATURES),
    all_features_dim=len(ALL_FEATURES),
    hidden_dim=64,
    num_layers=2,
    dropout=0.3
).to(device)

checkpoint = torch.load(f"{MODEL_PATH}/best_model.pt", weights_only=False, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Model loaded!")

## 2. Get Predictions

In [ ]:
# Get predictions (matching training notebook's evaluate function)
print("Getting predictions...")
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        vitals = batch['vitals'].to(device)
        labs = batch['labs'].to(device)
        labs_mask = batch['labs_mask'].to(device)
        all_features = batch['all_features'].to(device)
        labels = batch['label']
        
        outputs = model(vitals, labs, labs_mask, all_features)
        probs = outputs['probability'].squeeze()
        
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

y_true = np.array(all_labels)
y_prob = np.array(all_probs)

print(f"Predictions complete: {len(y_true):,} samples")
print(f"Positive labels: {y_true.sum():.0f} ({y_true.mean()*100:.1f}%)")
print(f"Prob range: [{y_prob.min():.4f}, {y_prob.max():.4f}]")

# Quick AUROC check - should be ~0.7263
from sklearn.metrics import roc_auc_score as _check_auroc
print(f"\nQuick AUROC check: {_check_auroc(y_true, y_prob):.4f} (expected ~0.7263)")

## 3. Calculate All Metrics

In [ ]:
# AUROC and AUPRC
auroc = roc_auc_score(y_true, y_prob)
auprc = average_precision_score(y_true, y_prob)

print("="*60)
print("PRIMARY METRICS")
print("="*60)
print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

In [ ]:
# Function to calculate metrics at a threshold
def calculate_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # Recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0  # Precision
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1 = 2 * (ppv * sensitivity) / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0
    
    return {
        'threshold': threshold,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'f1': f1,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
    }

In [ ]:
# Calculate metrics at various thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

print("\n" + "="*60)
print("METRICS AT VARIOUS THRESHOLDS")
print("="*60)

for thresh in thresholds:
    metrics = calculate_metrics_at_threshold(y_true, y_prob, thresh)
    results.append(metrics)
    print(f"\nThreshold: {thresh}")
    print(f"  Sensitivity (Recall): {metrics['sensitivity']:.4f}")
    print(f"  Specificity:          {metrics['specificity']:.4f}")
    print(f"  PPV (Precision):      {metrics['ppv']:.4f}")
    print(f"  NPV:                  {metrics['npv']:.4f}")
    print(f"  F1 Score:             {metrics['f1']:.4f}")

results_df = pd.DataFrame(results)

In [ ]:
# Find optimal threshold (maximize F1)
precisions, recalls, pr_thresholds = precision_recall_curve(y_true, y_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = pr_thresholds[optimal_idx] if optimal_idx < len(pr_thresholds) else 0.5

print(f"\nOptimal threshold (max F1): {optimal_threshold:.4f}")

optimal_metrics = calculate_metrics_at_threshold(y_true, y_prob, optimal_threshold)
print(f"\n" + "="*60)
print(f"METRICS AT OPTIMAL THRESHOLD ({optimal_threshold:.3f})")
print("="*60)
print(f"Sensitivity (Recall): {optimal_metrics['sensitivity']:.4f}")
print(f"Specificity:          {optimal_metrics['specificity']:.4f}")
print(f"PPV (Precision):      {optimal_metrics['ppv']:.4f}")
print(f"NPV:                  {optimal_metrics['npv']:.4f}")
print(f"F1 Score:             {optimal_metrics['f1']:.4f}")

In [ ]:
# Find threshold for high sensitivity (clinical use - catch most cases)
fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)

# Find threshold for 80% sensitivity
target_sensitivity = 0.80
idx_80 = np.argmin(np.abs(tpr - target_sensitivity))
threshold_80_sens = roc_thresholds[idx_80]

high_sens_metrics = calculate_metrics_at_threshold(y_true, y_prob, threshold_80_sens)

print(f"\n" + "="*60)
print(f"METRICS AT 80% SENSITIVITY (Clinical Use)")
print(f"Threshold: {threshold_80_sens:.4f}")
print("="*60)
print(f"Sensitivity (Recall): {high_sens_metrics['sensitivity']:.4f}")
print(f"Specificity:          {high_sens_metrics['specificity']:.4f}")
print(f"PPV (Precision):      {high_sens_metrics['ppv']:.4f}")
print(f"NPV:                  {high_sens_metrics['npv']:.4f}")
print(f"F1 Score:             {high_sens_metrics['f1']:.4f}")

## 4. Visualizations

In [ ]:
# Comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. ROC Curve
ax1 = axes[0, 0]
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'AUROC = {auroc:.4f}')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.scatter([1-high_sens_metrics['specificity']], [high_sens_metrics['sensitivity']], 
           color='red', s=100, zorder=5, label=f'80% Sensitivity')
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('ROC Curve', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Precision-Recall Curve
ax2 = axes[0, 1]
ax2.plot(recalls, precisions, 'g-', linewidth=2, label=f'AUPRC = {auprc:.4f}')
ax2.scatter([optimal_metrics['sensitivity']], [optimal_metrics['ppv']], 
           color='red', s=100, zorder=5, label=f'Optimal F1')
ax2.set_xlabel('Recall (Sensitivity)')
ax2.set_ylabel('Precision (PPV)')
ax2.set_title('Precision-Recall Curve', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Confusion Matrix at optimal threshold
ax3 = axes[0, 2]
y_pred_opt = (y_prob >= optimal_threshold).astype(int)
cm = confusion_matrix(y_true, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=['No Sepsis', 'Sepsis'],
            yticklabels=['No Sepsis', 'Sepsis'])
ax3.set_xlabel('Predicted')
ax3.set_ylabel('Actual')
ax3.set_title(f'Confusion Matrix (threshold={optimal_threshold:.2f})', fontweight='bold')

# 4. Metrics by Threshold
ax4 = axes[1, 0]
ax4.plot(results_df['threshold'], results_df['sensitivity'], 'b-o', label='Sensitivity')
ax4.plot(results_df['threshold'], results_df['specificity'], 'g-o', label='Specificity')
ax4.plot(results_df['threshold'], results_df['ppv'], 'r-o', label='PPV')
ax4.plot(results_df['threshold'], results_df['f1'], 'm-o', label='F1')
ax4.set_xlabel('Threshold')
ax4.set_ylabel('Score')
ax4.set_title('Metrics vs Threshold', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Probability Distribution
ax5 = axes[1, 1]
ax5.hist(y_prob[y_true==0], bins=50, alpha=0.5, label='No Sepsis', color='green')
ax5.hist(y_prob[y_true==1], bins=50, alpha=0.5, label='Sepsis', color='red')
ax5.axvline(x=optimal_threshold, color='black', linestyle='--', label=f'Threshold={optimal_threshold:.2f}')
ax5.set_xlabel('Predicted Probability')
ax5.set_ylabel('Count')
ax5.set_title('Prediction Distribution', fontweight='bold')
ax5.legend()

# 6. Summary Table
ax6 = axes[1, 2]
ax6.axis('off')
summary_text = f"""
COMPLETE METRICS SUMMARY
{'='*40}

Primary Metrics:
  AUROC:  {auroc:.4f}
  AUPRC:  {auprc:.4f}

At Optimal Threshold ({optimal_threshold:.3f}):
  Sensitivity: {optimal_metrics['sensitivity']:.4f}
  Specificity: {optimal_metrics['specificity']:.4f}
  PPV:         {optimal_metrics['ppv']:.4f}
  NPV:         {optimal_metrics['npv']:.4f}
  F1 Score:    {optimal_metrics['f1']:.4f}

At 80% Sensitivity ({threshold_80_sens:.3f}):
  Specificity: {high_sens_metrics['specificity']:.4f}
  PPV:         {high_sens_metrics['ppv']:.4f}
"""
ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/complete_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved to {OUTPUT_PATH}/complete_metrics.png")

## 5. Final Summary

In [ ]:
# Save complete metrics
complete_metrics = {
    'model_version': MODEL_VERSION,
    'n_test_samples': len(y_true),
    'n_positive': int(y_true.sum()),
    'prevalence': float(y_true.mean()),
    'primary_metrics': {
        'auroc': float(auroc),
        'auprc': float(auprc)
    },
    'optimal_threshold': {
        'threshold': float(optimal_threshold),
        'sensitivity': float(optimal_metrics['sensitivity']),
        'specificity': float(optimal_metrics['specificity']),
        'ppv': float(optimal_metrics['ppv']),
        'npv': float(optimal_metrics['npv']),
        'f1': float(optimal_metrics['f1'])
    },
    'high_sensitivity_threshold': {
        'threshold': float(threshold_80_sens),
        'sensitivity': float(high_sens_metrics['sensitivity']),
        'specificity': float(high_sens_metrics['specificity']),
        'ppv': float(high_sens_metrics['ppv']),
        'npv': float(high_sens_metrics['npv']),
        'f1': float(high_sens_metrics['f1'])
    },
    'metrics_by_threshold': results_df.to_dict('records')
}

with open(f"{OUTPUT_PATH}/complete_metrics.json", 'w') as f:
    json.dump(complete_metrics, f, indent=2)

print(f"Saved to {OUTPUT_PATH}/complete_metrics.json")

In [ ]:
# Print final summary table for documentation
print("\n" + "="*70)
print("COPY THIS TABLE TO YOUR DOCUMENTATION")
print("="*70)
print("\n| Metric | Value | Notes |")
print("|--------|-------|-------|")
print(f"| AUROC | {auroc:.4f} | Primary discrimination metric |")
print(f"| AUPRC | {auprc:.4f} | Better for imbalanced data |")
print(f"| Sensitivity | {optimal_metrics['sensitivity']:.4f} | At optimal threshold |")
print(f"| Specificity | {optimal_metrics['specificity']:.4f} | At optimal threshold |")
print(f"| PPV (Precision) | {optimal_metrics['ppv']:.4f} | At optimal threshold |")
print(f"| NPV | {optimal_metrics['npv']:.4f} | At optimal threshold |")
print(f"| F1 Score | {optimal_metrics['f1']:.4f} | At optimal threshold |")
print(f"| Optimal Threshold | {optimal_threshold:.4f} | Maximizes F1 |")